# Практика 2 — вариант 24
Соколовский Владислав, Ист-05/26б. Базовое извлечение.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import pandas as pd
load_dotenv()
assert os.getenv("GIGA_KEY"), "Создайте локальный .env с GIGA_KEY"
llm = GigaChat(credentials=os.environ["GIGA_KEY"], model="GigaChat-2", temperature=0.2, max_tokens=1000)
print(llm.invoke("Привет!").content)

In [ ]:
prompt = ChatPromptTemplate.from_messages([("system", "Ты эксперт по заявкам на аренду жилья."), ("human", "Заявка: {text}\nОпредели количество проживающих, включая детей. Верни только целое число.")])
chain = prompt | llm | StrOutputParser()
df = pd.read_csv("rental_24.csv", sep=";")
assert len(df) == 15
results = []
for t in df.text:
    try:
        results.append(chain.invoke({"text": t}))
    except Exception as e:
        results.append("ERROR: " + type(e).__name__)
df["result"] = results
display(df)

In [ ]:
df["correct"] = pd.to_numeric(df["result"], errors="coerce").eq(df["amount"])
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")
if df.result.str.startswith("ERROR:").all():
    print("API недоступен: точность не измерена")
else:
    print(f"Верно: {df.correct.sum()}; ошибок: {(~df.correct).sum()}; точность: {df.correct.mean():.1%}")
display(df.loc[~df.correct])

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from typing import Optional
class Rental(BaseModel):
    count: int = Field(ge=1)
    count_adults: Optional[int] = Field(default=None, ge=0)
    count_children: Optional[int] = Field(default=None, ge=0)
    start_date: Optional[str] = None
    nights: Optional[int] = Field(default=None, ge=0)
    price_per_day: Optional[int] = Field(default=None, ge=0)
    remarks: Optional[str] = None
parser = PydanticOutputParser(pydantic_object=Rental)
few = ChatPromptTemplate.from_messages([
    ("system", "Извлекай количество людей. Примеры: пара = 2; трое студентов = 3; семья с двумя детьми = 4; один человек и собака = 1. Не считай возраст, животных и комнаты. Верни число."),
    ("human", "{text}")])
improved = few | llm | StrOutputParser()
structured = ChatPromptTemplate.from_messages([
    ("system", "Извлекай данные аренды. Примеры: пара = 2 взрослых, 0 детей; 2 взрослых и ребенок = 3 человека; один человек и собака = 1, состав по возрасту неизвестен. Не угадывай возрастной состав. Дата без года в формате MM-DD, если точный день неизвестен — null. Для диапазона заезда бери раннюю дату, для длительности и цены максимум. При датах заезда и выезда вычисляй разницу в ночах; явный диапазон длительности важнее. Цена только за сутки, если единица неясна — null. Неизвестные поля null."),
    ("human", "Заявка: {text}\nВерни JSON. {format_instructions}")]).partial(format_instructions=parser.get_format_instructions())
json_chain = structured | llm | parser
def safe_invoke(c, t):
    try:
        return c.invoke({"text": t})
    except Exception as e:
        return "ERROR: " + type(e).__name__
df["improved"] = [safe_invoke(improved,t) for t in df.text]
df["improved_correct"] = pd.to_numeric(df.improved, errors="coerce").eq(df.amount)
df.to_csv("rental_improved_results.csv", index=False, encoding="utf-8-sig")
for name in ["result","improved"]:
    if df[name].str.startswith("ERROR:").all():
        print(name, "не измерено")
    else:
        print(name, pd.to_numeric(df[name], errors="coerce").eq(df.amount).mean())


In [ ]:
truth = pd.read_csv("rental_ground_truth.csv", sep=";")
assert len(truth) == len(df)
records = []
for t in df.text:
    result = safe_invoke(json_chain,t)
    records.append(result.model_dump() if isinstance(result,Rental) else {"error":result})
pred = pd.DataFrame(records).reindex(columns=["count","count_adults","count_children","start_date","nights","price_per_day","remarks","error"])
out = pd.concat([df[["text","amount"]], truth.add_prefix("expected_"), pred.add_prefix("pred_")],axis=1)
out.to_csv("rental_structured_results.csv", index=False, encoding="utf-8-sig")
fields = ["count_adults","count_children","start_date","nights","price_per_day"]
valid = pred.error.isna()
scores = {}
for f in fields:
    a,b = truth[f],pred[f]
    match = (a.eq(b) | (a.isna() & b.isna())) & valid
    scores[f] = match.mean()
if valid.any():
    print(pd.Series(scores).map(lambda x:f"{x:.1%}"))
    print("Средняя точность:", f"{sum(scores.values())/len(scores):.1%}")
else:
    print("Нет валидных JSON-ответов: точность не измерена")
display(out)